# Étape 1.3 — Où part le temps dans le PSO ? (issue #2)

**Objectif :** mesurer séparément les 5 morceaux d'une itération du PSO et en déduire lesquels coûtent cher.

| Morceau | Où | Outil de mesure |
|---|---|---|
| `kernelUpdateParticle` | GPU | `cudaEvent` |
| `kernelUpdatePBest` | GPU | `cudaEvent` |
| `cudaMemcpy` pBests GPU → CPU | PCIe | `cudaEvent` |
| boucle CPU de recherche du gBest | CPU | `std::chrono` |
| `cudaMemcpy` gBest CPU → GPU | PCIe | `cudaEvent` |

**Livrable :** un tableau des temps (ms et %). **Validé si :** la somme des 5 parties est égale au temps total à 5 % près.

Runtime : **GPU T4**.

In [ ]:
# 0 — Récupérer le dépôt (à chaque séance)
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
token   = userdata.get("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO} || (cd /content/{REPO} && git pull --rebase -q)
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout -B amina/1.3-chronometrage
!git log --oneline -3

On travaille sur **une branche** `amina/1.3-chronometrage`, comme prévu dans les règles du trinôme : `main` reste propre, et ton travail y entre par une *pull request* relue par Lounis.

### 1 — Comprendre `cudaEvent` (lis avant de coder)

Rappel du cours : un lancement de kernel est **asynchrone**. Le CPU continue sans attendre. On ne peut donc pas chronométrer un kernel avec une horloge CPU placée autour du lancement : on mesurerait seulement le temps de *lancer* le kernel (quelques µs), pas son exécution.

Un **cudaEvent** est un « top » déposé dans la file d'attente du GPU. Le GPU note l'heure quand il y arrive.

```cpp
cudaEvent_t e0, e1;
cudaEventCreate(&e0); cudaEventCreate(&e1);   // une seule fois, avant la boucle

cudaEventRecord(e0);                  // top de départ, mis dans la file du GPU
monKernel<<<blocs, threads>>>(...);   // le travail à mesurer
cudaEventRecord(e1);                  // top d'arrivée
cudaEventSynchronize(e1);             // le CPU attend que le GPU soit arrivé à e1
float ms;
cudaEventElapsedTime(&ms, e0, e1);    // temps GPU entre les deux tops, en millisecondes
```

Pour la partie **CPU** (la boucle du gBest), on utilise une horloge CPU classique :

```cpp
#include <chrono>
auto c0 = std::chrono::high_resolution_clock::now();
/* ... code CPU ... */
auto c1 = std::chrono::high_resolution_clock::now();
double ms = std::chrono::duration<double, std::milli>(c1 - c0).count();
```

❓ **Question à te poser :** `cudaEventSynchronize` oblige le CPU à attendre après chaque morceau. Est-ce que ça change le temps total du programme ? Garde ta réponse, on en parlera.

### 2 — Préparer les fichiers

On copie le PSO original dans un dossier de travail. **On ne modifie jamais `src/pso_original/`**, c'est notre référence.

In [ ]:
!mkdir -p /content/DE_CUDA/src/pso_timing
!cp /content/DE_CUDA/src/pso_original/{kernel.h,kernel.cpp,kernel.cu,main.cpp} /content/DE_CUDA/src/pso_timing/
%cd /content/DE_CUDA/src/pso_timing
!ls

### 3 — À toi : instrumenter `cuda_pso` dans `kernel.cu`

Ouvre `src/pso_timing/kernel.cu` dans l'explorateur de fichiers de Colab (icône 📁 à gauche, double-clic pour l'éditer). Fais **5 modifications** :

1. **En haut du fichier**, ajoute `#include <chrono>`.
2. **Avant la boucle `for (int iter…)`**, crée les deux events et les 5 compteurs :
   ```cpp
   cudaEvent_t e0, e1;
   cudaEventCreate(&e0); cudaEventCreate(&e1);
   float ms;
   double t_k1 = 0, t_k2 = 0, t_d2h = 0, t_cpu = 0, t_h2d = 0;
   auto T0 = std::chrono::high_resolution_clock::now();   // début du temps total
   ```
3. **Dans la boucle**, entoure chacun des 5 morceaux. Le premier est donné en exemple, **fais les 4 autres toi-même** sur le même modèle :
   ```cpp
   cudaEventRecord(e0);
   kernelUpdateParticle<<<blocksNum, threadsNum>>>(devPos, devVel, devPBest, devGBest,
                                                   getRandomClamped(), getRandomClamped());
   cudaEventRecord(e1); cudaEventSynchronize(e1);
   cudaEventElapsedTime(&ms, e0, e1); t_k1 += ms;
   ```
   - `kernelUpdatePBest` → `t_k2` (même modèle)
   - le `cudaMemcpy` de `pBests` (GPU → CPU) → `t_d2h` (même modèle)
   - la boucle `for (int i = 0; i < size; …)` → `t_cpu`, avec **`std::chrono`**
   - le `cudaMemcpy` de `gBest` (CPU → GPU) → `t_h2d` (même modèle)
4. **Juste après la boucle**, mesure le total et affiche le tableau :
   ```cpp
   auto T1 = std::chrono::high_resolution_clock::now();
   double total = std::chrono::duration<double, std::milli>(T1 - T0).count();
   double somme = t_k1 + t_k2 + t_d2h + t_cpu + t_h2d;
   printf("\n%-22s %10s %8s\n", "Morceau", "ms", "%");
   printf("%-22s %10.1f %7.1f%%\n", "kernelUpdateParticle", t_k1,  100*t_k1/total);
   printf("%-22s %10.1f %7.1f%%\n", "kernelUpdatePBest",    t_k2,  100*t_k2/total);
   printf("%-22s %10.1f %7.1f%%\n", "memcpy pBests D->H",   t_d2h, 100*t_d2h/total);
   printf("%-22s %10.1f %7.1f%%\n", "boucle CPU gBest",     t_cpu, 100*t_cpu/total);
   printf("%-22s %10.1f %7.1f%%\n", "memcpy gBest H->D",    t_h2d, 100*t_h2d/total);
   printf("%-22s %10.1f %7.1f%%\n", "SOMME des parties",    somme, 100*somme/total);
   printf("%-22s %10.1f\n",         "TOTAL boucle",         total);
   ```
5. **Avant les `cudaFree`**, libère les events : `cudaEventDestroy(e0); cudaEventDestroy(e1);`

Enregistre (Ctrl+S), puis compile et exécute la cellule suivante.

In [ ]:
%cd /content/DE_CUDA/src/pso_timing
!nvcc -O2 -arch=sm_75 -o pso_timing main.cpp kernel.cpp kernel.cu 2>&1 | grep -v "math_functions\|#warning\|\^~\|^ *|\|In file included\|^$"
!./pso_timing

**Si ça ne compile pas :** lis la **première** erreur (ligne et message), corrige, recommence. Colle-moi l'erreur exacte si tu bloques.

### 4 — Analyse (réponds dans le chat ou dans l'issue #2)

1. Quel morceau prend le plus de temps ? Est-ce du calcul GPU ?
2. La SOMME est-elle à moins de 5 % du TOTAL ? Sinon, où est passé le temps manquant ? (indice : `getRandomClamped()` et le lancement des kernels)
3. Compare le TOTAL avec les ~3 s mesurées à l'étape 1.1. Plus long, plus court ? Pourquoi ? (repense à la question sur `cudaEventSynchronize`)
4. Qu'est-ce que ça t'apprend pour la conception du DE sur GPU ?

### 5 — Sauvegarder et ouvrir une pull request

In [ ]:
%cd /content/DE_CUDA
!git add src/pso_timing
!git status --short
!git commit -m "1.3 : chronometrage cudaEvent du PSO (refs #2)" && git push -u origin amina/1.3-chronometrage

Ensuite, sur GitHub, un bandeau jaune **« Compare & pull request »** apparaît. Clique dessus, écris dans la description ton tableau de temps et tes réponses aux 4 questions, mets **Lounis** en *reviewer*, et crée la PR. Déplace la carte #2 dans **In Review** sur le tableau.

Quand Lounis approuve : **Merge** → l'issue #2 se ferme automatiquement si tu écris `closes #2` dans la description de la PR.